# Clean-Hour: when is electricity cleanest? (AI + Climate)
Can a forecast tell a household the cleanest hour to charge a car or run the dishwasher, and where does machine learning beat a simple rule like "run at midday"? We test five grids: **Virginia / Mid-Atlantic (our home region)**, California, Texas, Germany and Great Britain. Train 2023-2025, test every day of 2026 (never shuffled).

All numbers below come from the scripts in `src/`; this notebook runs them and shows the outputs.
**Honest scope:** the signal is the *average* carbon intensity of a whole regional grid, not marginal emissions and not one town. Savings are measured against starting at midnight (a deliberately naive baseline).

In [ ]:
import os, subprocess, sys
if not os.path.exists("src/multigrid.py"):
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    else:  # Colab
        subprocess.run(["git","clone","https://github.com/Harshal623/clean-hour.git"], check=False)
        os.chdir("clean-hour")
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], check=False)
def run(script, *args):
    r = subprocess.run([sys.executable,"-W","ignore",script,*args], capture_output=True, text=True)
    print(r.stdout[-12000:])
    if r.returncode:
        print("STDERR:", r.stderr[-2000:])
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200)
print(os.getcwd())

## 1. Data: five grids, Jan 2023 to Nov 2026
EIA-930 (Virginia, California, Texas), Energy-Charts (Germany), NESO (Great Britain), Open-Meteo weather forecasts. First run downloads about 400 MB and takes several minutes; later runs use the cache. Missing hours stay missing (never filled).

In [ ]:
run("src/grids.py")

## 2. The deciding test, all five grids
For a 4-hour job decided at 18:00 the evening before, with 12 / 24 / 48 / 72 hours of allowed delay: start at midnight vs persistence vs hour-of-week average vs our model vs perfect hindsight. The model predicts the *change* from the recent same-hour average (our first version predicted the level and was worse than persistence in California; see `results/multigrid_summary_direct.csv`).

In [ ]:
run("src/multigrid.py")

### Summary tables (from `results/multigrid_summary.csv`)

In [ ]:
S = pd.read_csv("results/multigrid_summary.csv")
for sl in (24, 72):
    print(f"--- job can wait {sl} h: gCO2/kWh of the chosen window (lower is better) and % cleaner than starting at midnight")
    x = S[S.slip_h == sl][["name","immediate","persistence","climatology","model","oracle","model_pct_saved","climatology_pct_saved","oracle_pct_saved"]]
    print(x.round(1).to_string(index=False))
    print("model minus hour-of-week rule (g/kWh), 95% block-bootstrap interval; negative = model better")
    print(S[S.slip_h == sl][["name","model_minus_climatology","model_minus_climatology_lo","model_minus_climatology_hi"]].round(1).to_string(index=False))
    print()
print("Great Britain vs NESO's own official forecast (model minus NESO, positive = NESO better):")
print(S[(S.grid=="gb")&S.model_minus_neso.notna()][["slip_h","neso","model","model_minus_neso","model_minus_neso_lo","model_minus_neso_hi"]].round(1).to_string(index=False))

In [ ]:
display(Image("results/multigrid_savings.png")) if os.path.exists("results/multigrid_savings.png") else None

## 3. Where it fails (three worst days per region)
Green band = our pick, grey band = best possible window. 12 of the 15 worst days fall in January to March. In Germany, Texas, Britain and Virginia the forecast error on the worst days is well above that region's average; in California it is not (and California's failures are small).

In [ ]:
run("src/failures.py")

In [ ]:
display(Image("results/failures_by_grid.png"))

## 4. Does it survive different emission factors? (`src/sensitivity.py`, about 10 minutes, saved output shown)
We recompute carbon intensity under five emission-factor sets and retrain. Great Britain is excluded because NESO supplies official intensity.

In [ ]:
E = pd.read_csv("results/ef_sensitivity_multigrid.csv")
print(E[E.slip_h==24].to_string(index=False))
print("
range of model-minus-rule at 24h per grid:"); print(E[E.slip_h==24].groupby("grid").model_minus_clim.agg(["min","max"]).round(1))

## 5. What it means for one household (`src/impact.py`, saved output shown)
Kilograms of CO2 avoided per year versus starting at midnight, assuming 3,000 kWh/yr of EV charging and 400 kWh/yr of laundry + dishwasher (our assumptions).

In [ ]:
I = pd.read_csv("results/household_impact.csv")
print(I[(I.strategy=="model")&(I.delay_h==24)][["name","load","kg_per_year","kg_per_year_lo","kg_per_year_hi"]].to_string(index=False))

## 6. Virginia deep dive: a better forecast is not automatically a better choice
From `results/experiments/` (Sidharth's analysis of the saved Virginia forecasts; reproduced here by `src/experiments/decision_stack.py`). A second-stage model trained to rank hours instead of predict levels captures 87-89% of the perfect-foresight saving versus 78-81% for the original model; scored on May-Nov 2026 only because Jan-Apr 2026 was used to train it.

In [ ]:
d = pd.read_csv("results/experiments/decision_stack.csv"); print(d.to_string(index=False))
display(Image("results/experiments/capture_vs_flexibility.png")); display(Image("results/experiments/clean_hours_heatmap.png"))

## 7. The app
`python app/app.py` opens the simple household app (pick region, appliance, how long it can wait, and a 2026 day). The older Loudoun data-center analysis is in `notebooks/virginia_deep_dive.ipynb`, `DECISION.md` and `docs/DATA_AUDIT.md`.